In [1]:
from pathlib import Path
from metasmith.python_api import *
from metasmith import examples
dtypes, contigs, references, transforms = examples.GenomicsAnnotation()

In [2]:
path_to_agent_home = Path("./cache/local_home").resolve()
smith = Agent(
    home = Source.FromLocal(path_to_agent_home),
)

In [3]:
task = smith.GenerateWorkflow(
    samples=[contigs],
    resources=[references],
    transforms=[transforms],
    targets=[
        dtypes["orf_annotations"].WithLineage([dtypes["contigs"]]),
    ]
)

In [ ]:
task.plans[0][0].RenderDAG("./dag", format="png")

In [1]:
from pathlib import Path
from metasmith.python_api import Agent, Source, Std, DataInstanceLibrary, TransformInstanceLibrary, WorkflowTask
from metasmith.python_api import DataTypeLibrary, Endpoint
from metasmith.python_api import Resources, Size, Duration
from local.constants import WORKSPACE_ROOT

# dtypes, containers, transforms = Std()

path_to_agent_home = Path("./cache/local_home").resolve()
smith = Agent(
    home = Source.FromLocal(path_to_agent_home),
)
# smith.Deploy()

In [9]:
for k in dtypes.types:
    print(k)

aa_sequences
contigs
oci_image_blast
oci_image_prodigal
orf_annotations
protein_reference_fasta


In [2]:
mock_types = DataTypeLibrary(
    types={
        k:Endpoint({k}|v)
        for k, v in [
            ("a", {"test"}),
            ("x", {"test"}),
            ("x_split", {"test"}),
            ("target", {"test"}),
            ("gathered", {"test"}),
        ]
    }
)

transforms = TransformInstanceLibrary("./transforms/simple_1", include_std=False)
transforms.AddTypeLibrary("mock", mock_types)
transforms.Save() # updates types
transforms.AddStub("no_op")
transforms.AddStub("no_op_no_fail")
transforms.AddStub("scatter")
transforms.AddStub("gather")
transforms.Save()

In [3]:
inputs = DataInstanceLibrary("./cache/dev21.mock.xgdb")
samples = []
for i in range(6):
    in_path = WORKSPACE_ROOT/f"main/local_mock/cache/test/mock_d.{i}"
    in_path.parent.mkdir(exist_ok=True)
    with open(in_path, "w") as f:
        f.write("2")
    inputs.AddTypeLibrary("mock", mock_types)
    t = "mock::a" if i%3==0 else "mock::x"
    # inputs.AddItem(in_path, "mock::a")
    # inputs.AddItem(in_path, "mock::x")
    inputs.AddItem(in_path, t)
    samples.append(in_path)
inputs.Save()
for p, n, e in inputs.Iterate():
    print(n, e, e.parents)

mock::a <[a,test]:q7XtOGUL> set()
mock::x <[test,x]:49oPy92Z> set()
mock::x <[test,x]:49oPy92Z> set()
mock::a <[a,test]:q7XtOGUL> set()
mock::x <[test,x]:49oPy92Z> set()
mock::x <[test,x]:49oPy92Z> set()


In [4]:
inputs.Consolidate()

{PosixPath('/home/tony/workspace/tools/Metasmith/main/local_mock/cache/test/mock_d.0'): PosixPath('1_mock_d.0'),
 PosixPath('/home/tony/workspace/tools/Metasmith/main/local_mock/cache/test/mock_d.1'): PosixPath('2_mock_d.1'),
 PosixPath('/home/tony/workspace/tools/Metasmith/main/local_mock/cache/test/mock_d.2'): PosixPath('3_mock_d.2'),
 PosixPath('/home/tony/workspace/tools/Metasmith/main/local_mock/cache/test/mock_d.3'): PosixPath('4_mock_d.3'),
 PosixPath('/home/tony/workspace/tools/Metasmith/main/local_mock/cache/test/mock_d.4'): PosixPath('5_mock_d.4'),
 PosixPath('/home/tony/workspace/tools/Metasmith/main/local_mock/cache/test/mock_d.5'): PosixPath('6_mock_d.5')}

In [ ]:
for loc, t,  in transforms.IterateTransforms():
    print(t.name, t.model)

In [ ]:
task = smith.GenerateWorkflow(
    samples    = [inputs.AsView({p}) for p in samples],
    resources  = [],
    transforms = [transforms],
    # targets    = [mock_types["b"]]
    targets    = [mock_types["target"]]
)
print(task.GetKey())

In [ ]:
# smith.StageWorkflow(task, on_exist='update_workflow', verify_external_paths=False)
smith.StageWorkflow(task, on_exist='clear', verify_external_paths=False)

In [ ]:
with open(WORKSPACE_ROOT/"secrets/slurm_account_fir") as f:
    SLURM_ACCOUNT = f.read()

params = dict(
    slurmAccount = SLURM_ACCOUNT,
    executor_queueSize = 100,
    process = dict(
        tries=3,
        array=5,
        cpus=1,
        memory='32 GB',
        time='12hours',
    ),
)
# smith.RunWorkflow(task, smith.GetNxfConfigPresets()["slurm"], params)

In [ ]:
smith.RunWorkflow(
    task="mVO8fJEE",
    config_file=smith.GetNxfConfigPresets()["local"],
    resource_overrides={
        # transforms
        "all": Resources(
            memory=Size.GB(3),
        ),

        # all of same transform
        transforms["no_op_no_fail"]: Resources(
            memory=Size.GB(3),
        ),

        # all of same transform in batch
        transforms["no_op"]: {
            1: Resources(
                memory=Size.GB(2),
            ),
        },

        # specifc step
        (1, 1): Resources(
            memory=Size.GB(2),
        ),
    },
)

In [ ]:
import re
re.match(r"^sample\s\d+,\s?step\s\d+$", "sample 1, step 1")